In [ ]:
import hashlib
from pathlib import Path
import sentence_transformers
import torch
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from tqdm.notebook import tqdm


# 1. Custom LangChain Embeddings
class LangChainE5Embeddings(Embeddings):

  def __init__(self, model_name: str = "intfloat/multilingual-e5-base"):
    self.device = "mps" if torch.backends.mps.is_available() else "cpu"
    self.model = sentence_transformers.SentenceTransformer(
        model_name, device=self.device
    )

  def embed_documents(self, texts: list[str]) -> list[list[float]]:
    prefixed = [f"passage: {doc}" for doc in texts]
    return self.model.encode(
        prefixed,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

  def embed_query(self, text: str) -> list[float]:
    return self.model.encode(
        f"query: {text}", normalize_embeddings=True, convert_to_numpy=True
    ).tolist()


# Helper function to chunk text
def chunk_text_file(
    text: str, chunk_size_words: int = 200, overlap_words: int = 30
) -> list[str]:
  words = text.split()
  if not words:
    return []
  chunks = []
  for i in range(0, len(words), chunk_size_words - overlap_words):
    chunk_words = words[i : i + chunk_size_words]
    chunks.append(" ".join(chunk_words))
    if i + chunk_size_words >= len(words):
      break
  return chunks


# 2. Setup Vector Store
BASE_DIR = Path.cwd()
MAIN_CATEGORIES = [
    "أحكام التمييز",
    "المحكمة الدستورية",
    "هيئة التشريع والرأي القانوني",
    "النيابة العامة"
]

embeddings = LangChainE5Embeddings()

vector_db = Chroma(
    collection_name="legal_documents",
    embedding_function=embeddings,
    persist_directory="./legal_chroma_db2",
    collection_metadata={"hnsw:space": "cosine"},
)

BUFFER_SIZE = 2000
docs_buffer, meta_buffer, ids_buffer = [], [], []
total_indexed = 0

# 3. Processing Loop with MD5 Unique Hashing
for category in MAIN_CATEGORIES:
  cat_path = BASE_DIR / category
  if not cat_path.exists():
    print(f"Directory not found: {cat_path}")
    continue

  file_list = list(cat_path.rglob("*.txt"))
  print(f"\nProcessing {len(file_list)} files in [{category}]...")

  for file_path in tqdm(file_list, desc=category):
    try:
      with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

      chunks = chunk_text_file(content)
      rel_path = str(file_path.relative_to(cat_path))

      for idx, chunk in enumerate(chunks):
        # Uniquely hash Category + Full Relative Path + Chunk Index
        unique_seed = f"{category}_{rel_path}_{idx}"
        chunk_id = hashlib.md5(unique_seed.encode("utf-8")).hexdigest()

        docs_buffer.append(chunk)
        meta_buffer.append({
            "source_file": file_path.name,
            "main_category": category,
            "sub_path": rel_path,
            "chunk_index": idx,
        })
        ids_buffer.append(chunk_id)

        # Flush to database when buffer reaches size limit
        if len(docs_buffer) >= BUFFER_SIZE:
          vector_db.add_texts(
              texts=docs_buffer, metadatas=meta_buffer, ids=ids_buffer
          )
          total_indexed += len(docs_buffer)
          docs_buffer.clear()
          meta_buffer.clear()
          ids_buffer.clear()

    except Exception as e:
      print(f"Error reading file {file_path}: {e}")

# Flush remaining items
if docs_buffer:
  vector_db.add_texts(texts=docs_buffer, metadatas=meta_buffer, ids=ids_buffer)
  total_indexed += len(docs_buffer)

print(
    f"\nDone! Successfully embedded and indexed {total_indexed} total chunks."
)
print(f"Total Database Count: {vector_db._collection.count()} chunks.")

## for adding new files

In [ ]:
import hashlib
from pathlib import Path
import sentence_transformers
import torch
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from tqdm.notebook import tqdm

# 1. Custom LangChain Embeddings
class LangChainE5Embeddings(Embeddings):

  def __init__(self, model_name: str = "intfloat/multilingual-e5-base"):
    self.device = "mps" if torch.backends.mps.is_available() else "cpu"
    self.model = sentence_transformers.SentenceTransformer(
        model_name, device=self.device
    )

  def embed_documents(self, texts: list[str]) -> list[list[float]]:
    prefixed = [f"passage: {doc}" for doc in texts]
    return self.model.encode(
        prefixed,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).tolist()

  def embed_query(self, text: str) -> list[float]:
    return self.model.encode(
        f"query: {text}", normalize_embeddings=True, convert_to_numpy=True
    ).tolist()


# Helper function to chunk text
def chunk_text_file(
    text: str, chunk_size_words: int = 200, overlap_words: int = 30
) -> list[str]:
  words = text.split()
  if not words:
    return []
  chunks = []
  for i in range(0, len(words), chunk_size_words - overlap_words):
    chunk_words = words[i : i + chunk_size_words]
    chunks.append(" ".join(chunk_words))
    if i + chunk_size_words >= len(words):
      break
  return chunks


# 2. Setup Vector Store
BASE_DIR = Path.cwd()
MAIN_CATEGORIES = [
    "أحكام التمييز",
    "المحكمة الدستورية",
    "هيئة التشريع والرأي القانوني",
]

embeddings = LangChainE5Embeddings()

# Initialize Chroma Store
vector_db = Chroma(
    collection_name="legal_documents",
    embedding_function=embeddings,
    persist_directory="./legal_chroma_db",
    collection_metadata={"hnsw:space": "cosine"},
)

# 1. Fetch all existing IDs currently stored in ChromaDB
existing_ids = set(vector_db._collection.get()["ids"])
print(f"Current DB contains {len(existing_ids)} existing chunk IDs.")

BUFFER_SIZE = 2000
docs_buffer, meta_buffer, ids_buffer = [], [], []
total_new_indexed = 0

for category in MAIN_CATEGORIES:
  cat_path = BASE_DIR / category
  if not cat_path.exists():
    continue

  file_list = list(cat_path.rglob("*.txt"))
  print(f"\nChecking [{category}] for new files/chunks...")

  for file_path in tqdm(file_list, desc=category):
    try:
      rel_path = str(file_path.relative_to(cat_path))

      # Quick check: Generate chunk_0 ID to see if file was already indexed
      sample_id = hashlib.md5(f"{category}_{rel_path}_0".encode()).hexdigest()
      if sample_id in existing_ids:
        continue  # Skip entire file if chunk_0 already exists in DB

      # If it's a NEW file, read and chunk it
      with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

      chunks = chunk_text_file(content)

      for idx, chunk in enumerate(chunks):
        unique_seed = f"{category}_{rel_path}_{idx}"
        chunk_id = hashlib.md5(unique_seed.encode("utf-8")).hexdigest()

        # Double check chunk ID isn't in DB
        if chunk_id not in existing_ids:
          docs_buffer.append(chunk)
          meta_buffer.append({
              "source_file": file_path.name,
              "main_category": category,
              "sub_path": rel_path,
              "chunk_index": idx,
          })
          ids_buffer.append(chunk_id)
          existing_ids.add(chunk_id)  # Track in local set

        # Flush buffer when full
        if len(docs_buffer) >= BUFFER_SIZE:
          vector_db.add_texts(
              texts=docs_buffer, metadatas=meta_buffer, ids=ids_buffer
          )
          total_new_indexed += len(docs_buffer)
          docs_buffer.clear()
          meta_buffer.clear()
          ids_buffer.clear()

    except Exception as e:
      print(f"Error reading file {file_path}: {e}")

# Flush remaining new items
if docs_buffer:
  vector_db.add_texts(texts=docs_buffer, metadatas=meta_buffer, ids=ids_buffer)
  total_new_indexed += len(docs_buffer)

print(
    f"\nDone! Added {total_new_indexed} new chunks to the database without"
    " re-embedding existing files."
)

## Chatbot

In [ ]:
import os
import torch
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer

os.environ["GROQ_API_KEY"] = "api"

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq

In [ ]:
class BahrainLegalChatbot:

  def __init__(self, vectorstore, llm, prompt):
    self.vectorstore = vectorstore
    self.llm = llm
    self.prompt = prompt
    self.chat_history = []

  def ask(self, user_query: str, selected_category: str = "All") -> str:
    # Build filter parameter for ChromaDB
    search_filter = None
    if selected_category != "All":
      search_filter = {"category": selected_category}

    # Retrieve similarity matches using the category filter
    docs = self.vectorstore.similarity_search(
        query=user_query, k=5, filter=search_filter
    )

    fallback_msg = (
        "عذراً، لا تتوفر لدي معلومات كافية في المستندات القانونية المتاحة"
        " للإجابة على هذا السؤال."
    )

    if not docs:
      return fallback_msg

    context_text = ""
    sources = set()
    for doc in docs:
      source_file = doc.metadata.get("source_file", "مستند غير معروف")
      chunk_idx = doc.metadata.get("chunk_index", "")
      sources.add(f"{source_file} (جزء {chunk_idx})")
      context_text += (
          f"\n---\nالمستند: {source_file}\nالنص:\n{doc.page_content}\n"
      )

    chain = self.prompt | self.llm
    response = chain.invoke({
        "context": context_text,
        "chat_history": self.chat_history,
        "question": user_query,
    })

    response_text = response.content.strip()

    self.chat_history.append(HumanMessage(content=user_query))
    self.chat_history.append(AIMessage(content=response_text))

    if fallback_msg in response_text or "لا تتوفر لدي معلومات" in response_text:
      return response_text

    sources_formatted = "\n".join([f"- {s}" for s in sources])
    return f"{response_text}\n\n**المصادر والمراجع:**\n{sources_formatted}"

In [ ]:
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)

LEGAL_SYSTEM_PROMPT = """أنت مستشار قانوني خبير ومتخصص في التشريعات والقوانين الخاصة بمملكة البحرين.
قم بالتحليل الدقيق للنصوص القانونية المرفقة واستخراج الإجابة المباشرة عن سؤال المستخدم.

**تعليمات الرد:**
1. إذا كانت الإجابة موجودة في أي جزء من النصوص المرفقة، استخرج النص وشرحه بوضوح وموجز كافٍ دون إطالة.
2. إذا لم تجد الإجابة إطلاقاً في النصوص المرفقة، أجب حصراً بـ: "عذراً، لا تتوفر لدي معلومات كافية في المستندات القانونية المتاحة للإجابة على هذا السؤال."
3. الهيكل المطلوب للإجابة الإلزامي:
   - **النص القانوني / المادة المباشرة**
   - **الشرح والتطبيق القانوني**
   - **ملخص تنفيذي موجز** (سطران فقط يلخصان الإجابة بشكل مركز)
   - **3 أسئلة مقترحة ذات صلة**

النصوص القانونية المرفقة:
{context}"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", LEGAL_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

In [ ]:
import re


def normalize_arabic(text: str) -> str:
  """Normalizes Arabic letters for consistent matching."""
  text = re.sub(r"[\u064B-\u0652]", "", text)  # Remove diacritics
  text = re.sub(r"[إأآ]", "ا", text)  # Standardize Alif
  text = re.sub(r"ى", "ي", text)  # Standardize Alif Maqsura
  text = re.sub(r"ة", "ه", text)  # Standardize Ta Marbuta
  return text.strip()


class BahrainLegalChatbot:

  def __init__(self, vectorstore, llm, prompt):
    self.vectorstore = vectorstore
    self.llm = llm
    self.prompt = prompt
    self.chat_history = []

  def ask(self, user_query: str, category_filter: str = "الكل") -> str:
    # Build filter dictionary
    filter_dict = None
    if category_filter != "الكل":
      filter_dict = {"main_category": category_filter}

    # Retrieve top 12 chunks to maximize semantic coverage
    docs = self.vectorstore.similarity_search(
        query=user_query, k=12, filter=filter_dict
    )

    # Secondary fuzzy check for short search terms
    clean_query = normalize_arabic(user_query)
    if len(clean_query.split()) <= 3:
      extra_docs = self.vectorstore.similarity_search(
          query=user_query, k=30, filter=filter_dict
      )
      matched_docs = [
          doc
          for doc in extra_docs
          if clean_query in normalize_arabic(doc.page_content)
      ]
      if matched_docs:
        docs = matched_docs[:12]

    fallback_msg = (
        "عذراً، لا تتوفر لدي معلومات كافية في المستندات القانونية المتاحة"
        " للإجابة على هذا السؤال."
    )

    if not docs:
      return fallback_msg

    # Build context string
    context_text = ""
    sources = set()
    for doc in docs:
      source_file = doc.metadata.get("source_file", "مستند غير معروف")
      chunk_idx = doc.metadata.get("chunk_index", "")
      cat = doc.metadata.get("main_category", "")
      sources.add(
          f"[{cat}] {source_file} (جزء {chunk_idx})"
          if cat
          else f"{source_file} (جزء {chunk_idx})"
      )
      context_text += (
          f"\n---\nالمستند: {source_file}\nالنص:\n{doc.page_content}\n"
      )

    # Invoke LLM chain
    chain = self.prompt | self.llm
    response = chain.invoke({
        "context": context_text,
        "chat_history": self.chat_history,
        "question": user_query,
    })

    response_text = response.content.strip()

    # Save conversation state
    self.chat_history.append(HumanMessage(content=user_query))
    self.chat_history.append(AIMessage(content=response_text))

    if fallback_msg in response_text or "لا تتوفر لدي معلومات" in response_text:
      return response_text

    sources_formatted = "\n".join([f"- {s}" for s in sources])
    return f"{response_text}\n\n**المصادر والمراجع:**\n{sources_formatted}"

In [ ]:
# INTERACTIVE TERMINAL LOOP WITH CATEGORY SELECTION
if __name__ == "__main__":
  bot = BahrainLegalChatbot(vector_db, llm, prompt_template)

  # Category Options Menu
  CATEGORIES = {
    "1": "الكل",
    "2": "أحكام التمييز",
    "3": "المحكمة الدستورية",
    "4": "هيئة التشريع والرأي القانوني",
}

  print("=" * 65)
  print(
      "المساعد القانوني لمملكة البحرين جاهز للاستخدام. (اكتب 'خروج' لإغلاق"
      " المحادثة)"
  )
  print("=" * 65)

  # Interactive Folder Selection
  print("\nاختر نطاق البحث المطلوب:")
  print("1. البحث في جميع الفئات (الكل)")
  print("2. أحكام التمييز")
  print("3. المحكمة الدستورية")
  print("4. هيئة التشريع والرأي القانوني")

  choice = input("\nأدخل رقم الاختيار (1-4) [الافتراضي: 1]: ").strip()
  selected_category = CATEGORIES.get(choice, "الكل")

  print(f"\n تم ضبط نطاق البحث على: [{selected_category}]")
  print("-" * 65)

  while True:
    user_input = input("\nالمستخدم: ")
    if user_input.strip().lower() in ["exit", "quit", "خروج"]:
      print("تم إغلاق المحادثة.")
      break

    if not user_input.strip():
      continue

    print(
        f"\nجاري البحث في ({selected_category}) وتحليل النصوص القانونية..."
    )
    answer = bot.ask(user_input, category_filter=selected_category)
    print(f"\nالمساعد القانوني:\n{answer}")